### Clone repo and setup

In [1]:
!git clone https://github.com/ThuyHaLE/FRAS.git
%cd FRAS

Cloning into 'FRAS'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 230 (delta 105), reused 196 (delta 74), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 1.74 MiB | 20.94 MiB/s, done.
Resolving deltas: 100% (105/105), done.


In [3]:
!pip -q install -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 14.6 MB/s eta 0:00:00


In [3]:
import pandas as pd
from features import BASE_FEATURES, TARGET_TIME, TARGET_EVENT
from models.utils.enrich_features import enrich_features
from models.utils.filter_features import _TrainerWrapper, filter_features
from models.strategies import (
    ModelStrategy,
    RSFStrategy,
    GradBoostStrategy,
    CoxnetStrategy,
    CGBStrategy,
    _STRATEGY_REGISTRY)
from models import (
    # Trainer
    SurvivalTrainer,
    # Ensemble
    EnsembleTrainer, TrainerConfig,
    # Feature selection
    PermutationImportance, SurvivalModelWrapper,
    filter_features, filter_features_independent,
    TierThresholds, DEFAULT_THRESHOLDS, COXNET_THRESHOLDS,
)

### 1. Single trainer — string (backward compat)

In [6]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

trainer = SurvivalTrainer(feature_cols=BASE_FEATURES, model_type="rsf")
trainer.cross_validate(train_df)
trainer.fit(train_df)
df_sub = trainer.make_submission(test_df)

df_sub

[auto_params] n=221  [RSF] trees=265  max_features='sqrt'  leaf=11  split=22

── Fold 1/5 [RSF] ──
   C-index@48h: 0.9423  Hybrid Score: 0.9262

── Fold 2/5 [RSF] ──
   C-index@48h: 0.9559  Hybrid Score: 0.9367

── Fold 3/5 [RSF] ──
   C-index@48h: 0.9464  Hybrid Score: 0.9412

── Fold 4/5 [RSF] ──
   C-index@48h: 0.9187  Hybrid Score: 0.9157

── Fold 5/5 [RSF] ──
   C-index@48h: 0.9135  Hybrid Score: 0.9376

[RSF] Fold scores          : [0.9262  0.9367  0.9412  0.9157  0.9376]
[RSF] Mean CV Hybrid Score : 0.9315 ± 0.0094

=== OOF evaluation ===

 Horizon    C-index    Brier raw    Brier cal
------------------------------------------------
    12h      0.9079       0.0842       0.0802
    24h      0.9269       0.0862       0.0685  (×0.3)
    48h      0.9274       0.0880       0.0646  (×0.4)
    72h      0.9309       0.1010       0.0796  (×0.3)
------------------------------------------------
                  Hybrid Score  0.9290   = 0.3×C@48h + 0.7×(1−WBS)

=== Retraining [RSF] on ful

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.010851,0.020808,0.025672,0.250247
1,13353600,0.234325,0.613990,0.669699,0.677095
2,13942327,0.009261,0.016963,0.022683,0.238819
3,16112781,0.236515,0.639422,0.690951,0.706306
4,17132808,0.769115,0.800152,0.803548,0.803548
...,...,...,...,...,...
90,94627327,0.146561,0.163771,0.206309,0.391372
91,96570675,0.006967,0.012688,0.014440,0.175264
92,97225766,0.013203,0.027838,0.031360,0.253864
93,98446281,0.103625,0.121439,0.142705,0.335333


### 2. Single trainer — inject strategy

In [10]:
# ── 2. Single trainer — inject strategy ──────────────────────
trainer = SurvivalTrainer(
    feature_cols = BASE_FEATURES,
    strategy     = RSFStrategy(n_trees=300, max_features="log2"),
)
trainer.cross_validate(train_df)
trainer.fit(train_df)

df_sub = trainer.make_submission(test_df)

df_sub

[auto_params] n=221  [RSF] trees=265  max_features='log2'  leaf=11  split=22

── Fold 1/5 [RSF] ──
   C-index@48h: 0.9423  Hybrid Score: 0.9262

── Fold 2/5 [RSF] ──
   C-index@48h: 0.9559  Hybrid Score: 0.9367

── Fold 3/5 [RSF] ──
   C-index@48h: 0.9464  Hybrid Score: 0.9412

── Fold 4/5 [RSF] ──
   C-index@48h: 0.9187  Hybrid Score: 0.9157

── Fold 5/5 [RSF] ──
   C-index@48h: 0.9135  Hybrid Score: 0.9376

[RSF] Fold scores          : [0.9262  0.9367  0.9412  0.9157  0.9376]
[RSF] Mean CV Hybrid Score : 0.9315 ± 0.0094

=== OOF evaluation ===

 Horizon    C-index    Brier raw    Brier cal
------------------------------------------------
    12h      0.9079       0.0842       0.0802
    24h      0.9269       0.0862       0.0685  (×0.3)
    48h      0.9274       0.0880       0.0646  (×0.4)
    72h      0.9309       0.1010       0.0796  (×0.3)
------------------------------------------------
                  Hybrid Score  0.9290   = 0.3×C@48h + 0.7×(1−WBS)

=== Retraining [RSF] on ful

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.010851,0.020808,0.025672,0.250247
1,13353600,0.234325,0.613990,0.669699,0.677095
2,13942327,0.009261,0.016963,0.022683,0.238819
3,16112781,0.236515,0.639422,0.690951,0.706306
4,17132808,0.769115,0.800152,0.803548,0.803548
...,...,...,...,...,...
90,94627327,0.146561,0.163771,0.206309,0.391372
91,96570675,0.006967,0.012688,0.014440,0.175264
92,97225766,0.013203,0.027838,0.031360,0.253864
93,98446281,0.103625,0.121439,0.142705,0.335333


### 3. Add new model strategy (XGBSEDebiasedBCE)

In [14]:
!pip install numpy>=2.0
!pip -q install xgbse

In [25]:
from xgbse import XGBSEDebiasedBCE
import numpy as np
from typing import Any
from dataclasses import field

class _XGBSurvivalFn:
    """
    Wrap one row of XGBSE predict DataFrame into a callable with .domain,
    similar to the sksurv step function interface that _survival_to_hit_probs expects.
    """
    __slots__ = ("times", "surv", "domain")

    def __init__(self, times: np.ndarray, surv: np.ndarray):
        self.times  = times
        self.surv   = np.clip(surv, 0.0, 1.0)
        self.domain = (float(times[0]), float(times[-1]))

    def __call__(self, t: float) -> float:
        return float(np.interp(t, self.times, self.surv,
                               left=self.surv[0], right=self.surv[-1]))

class XGBSEStrategy(ModelStrategy):
    """
    XGBSEDebiasedBCE survival strategy.
    Self resolve time_bins from training labels in fit().
    """

    def __init__(
        self,
        n_estimators:     int   = 200,
        learning_rate:    float = 0.05,
        max_depth:        int   = 4,
        subsample:        float = 0.8,
        colsample_bytree: float = 0.8,
        n_time_bins:      int   = 20,
    ):
        self.n_estimators     = n_estimators
        self.learning_rate    = learning_rate
        self.max_depth        = max_depth
        self.subsample        = subsample
        self.colsample_bytree = colsample_bytree
        self.n_time_bins      = n_time_bins
        self._time_bins       = None

    """
    XGBSEDebiasedBCE survival strategy.

    Self resolve time_bins from training labels in fit(),
    do not require user to specify
    """

    @property
    def name(self) -> str:
        return "XGBSE"

    # ── auto_params: scale by dataset size──────────────
    def auto_params(self, n: int) -> None:
        self.n_estimators = int(min(500, max(100, n * 1.0)))
        self.n_time_bins  = max(10, min(30, n // 10))

    # ── build: init model not fit ───────────────────────────────
    def build(self, random_state: int) -> XGBSEDebiasedBCE:
        return XGBSEDebiasedBCE(
            xgb_params={
                "n_estimators":     self.n_estimators,
                "learning_rate":    self.learning_rate,
                "max_depth":        self.max_depth,
                "subsample":        self.subsample,
                "colsample_bytree": self.colsample_bytree,
                "use_label_encoder": False,
                "eval_metric":      "logloss",
                "random_state":     random_state,
            }
        )

    # ── fit: resolve time_bins from label, then fit ─────────────────────
    def fit(self, model: XGBSEDebiasedBCE, X: np.ndarray, y: Any) -> Any:
        # y is structured array from Surv.from_dataframe
        # default field names of sksurv: y.dtype.names[0]=event, [1]=time
        event_field, time_field = y.dtype.names
        times  = y[time_field]
        events = y[event_field].astype(bool)

        # Use percentiles of event times to create bins (skip censored)
        event_times = times[events] if events.any() else times
        self._time_bins = np.unique(
            np.percentile(event_times,
                          np.linspace(0, 100, self.n_time_bins + 1))
        )

        model.fit(X, y, time_bins=self._time_bins)
        return model

    # ── predict_survival: return list callable with .domain ───────────────
    def predict_survival(self, model: XGBSEDebiasedBCE, X: np.ndarray) -> list:
        df: pd.DataFrame = model.predict(X)          # (n_samples, n_times)
        times = np.array(df.columns, dtype=float)    # time points from training

        # XGBSE returns survival P(T > t), NOT CDF
        # → use directly, do not need 1 - df
        surv_matrix = df.values                      # shape (n, T)

        return [
            _XGBSurvivalFn(times, surv_matrix[i])
            for i in range(len(surv_matrix))
        ]

    def summary_str(self) -> str:
        bins_str = (
            f"[{self._time_bins[0]:.0f}…{self._time_bins[-1]:.0f}]"
            if self._time_bins is not None else "not fit yet"
        )
        return (
            f"[XGBSE] n_est={self.n_estimators}  lr={self.learning_rate}  "
            f"depth={self.max_depth}  time_bins={len(self._time_bins) if self._time_bins is not None else '?'} {bins_str}"
        )

trainer = SurvivalTrainer(
    feature_cols=BASE_FEATURES,
    strategy=XGBSEStrategy(n_estimators=200, learning_rate=0.05),
)
trainer.cross_validate(train_df)
trainer.fit(train_df)
df_sub = trainer.make_submission(test_df)

df_sub

[auto_params] n=221  [XGBSE] n_est=221  lr=0.05  depth=4  time_bins=? not fit yet

── Fold 1/5 [XGBSE] ──
   C-index@48h: 0.8976  Hybrid Score: 0.9419

── Fold 2/5 [XGBSE] ──
   C-index@48h: 0.8978  Hybrid Score: 0.9633

── Fold 3/5 [XGBSE] ──
   C-index@48h: 0.9320  Hybrid Score: 0.9663

── Fold 4/5 [XGBSE] ──
   C-index@48h: 0.9253  Hybrid Score: 0.9703

── Fold 5/5 [XGBSE] ──
   C-index@48h: 0.8847  Hybrid Score: 0.9582

[XGBSE] Fold scores          : [0.9419  0.9633  0.9663  0.9703  0.9582]
[XGBSE] Mean CV Hybrid Score : 0.9600 ± 0.0099

=== OOF evaluation ===

 Horizon    C-index    Brier raw    Brier cal
------------------------------------------------
    12h      0.9352       0.0632       0.0624
    24h      0.9148       0.0395       0.0302  (×0.3)
    48h      0.9108       0.0280       0.0179  (×0.4)
    72h      0.9127       0.0109       0.0047  (×0.3)
------------------------------------------------
                  Hybrid Score  0.9609   = 0.3×C@48h + 0.7×(1−WBS)

=== Retr

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.003109,0.003109,0.004251,0.149865
1,13353600,0.794852,0.986597,0.994902,0.994902
2,13942327,0.003109,0.003109,0.004251,0.149865
3,16112781,0.432540,0.788176,0.941405,0.941405
4,17132808,0.008817,0.008817,0.010598,0.197278
...,...,...,...,...,...
90,94627327,0.005501,0.005501,0.006327,0.161239
91,96570675,0.003109,0.003109,0.004251,0.149865
92,97225766,0.003109,0.003109,0.004251,0.149865
93,98446281,0.005501,0.005501,0.006327,0.161239


### 4. Ensemble

In [26]:
configs = [
    TrainerConfig("RSF",    RSFStrategy(n_trees=300),  BASE_FEATURES),
    TrainerConfig("GB",     GradBoostStrategy(),       BASE_FEATURES),
    TrainerConfig("Coxnet", CoxnetStrategy(),          BASE_FEATURES),
    TrainerConfig("CGB",    CGBStrategy(),             BASE_FEATURES),
]
ensemble = EnsembleTrainer(configs)
ensemble.cross_validate(train_df)
ensemble.optimize_weights(train_df, strategy="scipy")

# So sánh tổ hợp con (optional)
results = ensemble.compare_subsets(train_df, subset_sizes=[2, 3])
ensemble.apply_best_subset(results["best"], train_df)

ensemble.fit(train_df)
df_sub = ensemble.make_submission(test_df)

df_sub

  EnsembleTrainer: RSF cross_validate
[auto_params] n=221  [RSF] trees=265  max_features='sqrt'  leaf=11  split=22

── Fold 1/5 [RSF] ──
   C-index@48h: 0.9423  Hybrid Score: 0.9262

── Fold 2/5 [RSF] ──
   C-index@48h: 0.9559  Hybrid Score: 0.9367

── Fold 3/5 [RSF] ──
   C-index@48h: 0.9464  Hybrid Score: 0.9412

── Fold 4/5 [RSF] ──
   C-index@48h: 0.9187  Hybrid Score: 0.9157

── Fold 5/5 [RSF] ──
   C-index@48h: 0.9135  Hybrid Score: 0.9376

[RSF] Fold scores          : [0.9262  0.9367  0.9412  0.9157  0.9376]
[RSF] Mean CV Hybrid Score : 0.9315 ± 0.0094
  EnsembleTrainer: GB cross_validate
[auto_params] n=221  [GradBoost] trees=176  lr=0.05  depth=2  sub=0.75  val_frac=0.20

── Fold 1/5 [GradBoost] ──
   C-index@48h: 0.9602  Hybrid Score: 0.9672

── Fold 2/5 [GradBoost] ──
   C-index@48h: 0.9549  Hybrid Score: 0.9800

── Fold 3/5 [GradBoost] ──
   C-index@48h: 0.9485  Hybrid Score: 0.9725

── Fold 4/5 [GradBoost] ──
   C-index@48h: 0.9341  Hybrid Score: 0.9724

── Fold 5/5 [GradB

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.005358,0.008492,0.008492,0.164179
1,13353600,0.757028,0.959969,0.974555,0.974555
2,13942327,0.009147,0.013995,0.013995,0.198850
3,16112781,0.717372,0.951201,0.969054,0.969054
4,17132808,0.037468,0.053109,0.053109,0.273314
...,...,...,...,...,...
90,94627327,0.024769,0.037004,0.037004,0.260528
91,96570675,0.008863,0.013725,0.013725,0.198231
92,97225766,0.004764,0.007686,0.007686,0.156893
93,98446281,0.011706,0.019200,0.019200,0.227639


### 5. Feature selection — string (backward compat)

In [4]:
train_df = pd.read_csv('data/train.csv')
enriched_train_df = enrich_features(pd.read_csv('data/train.csv'))

enriched_features = list(set(enriched_train_df.columns) - set(train_df.columns))
enriched_features

['momentum_threat',
 'log1p_dist_min',
 'eta_radial',
 'area_speed_ratio',
 'hour_sin',
 'dist_change_norm',
 'radial_bearing_threat',
 'closing_neg',
 'log1p_reliable_slope',
 'log1p_speed_consistency',
 'eta_combined',
 'linear_eta_ci',
 'log1p_centroid_disp',
 'log1p_projected',
 'eta_projected',
 'eta_closing',
 'area_radial_threat',
 'dist_cv',
 'reliable_accel',
 'hour_cos',
 'is_approaching',
 'reliable_eta_ci',
 'eta_bearing_adjusted',
 'directional_growth',
 'dt_x_alignment',
 'log1p_dist_change',
 'log1p_dist_slope',
 'log1p_threat_alignment',
 'log1p_dist_std',
 'eta_along_track',
 'growth_pressure',
 'log1p_radial_growth_rate',
 'log1p_radial_growth',
 'reliable_slope',
 'log1p_centroid_speed',
 'aligned_along_speed',
 'relative_accel',
 'bearing_threat_factor',
 'closing_pos',
 'dist_per_area',
 'log1p_closing_speed',
 'is_high_wind_hour',
 'eta_aligned_along',
 'speed_consistency',
 'alignment_x_nperim',
 'threat_alignment',
 'cross_track_ratio']

In [5]:
remove_feature = ['dist_min_ci_0_5h']
base_features = [f for f in BASE_FEATURES if f not in remove_feature]
base_features

['num_perimeters_0_5h',
 'dt_first_last_0_5h',
 'low_temporal_resolution_0_5h',
 'area_first_ha',
 'area_growth_abs_0_5h',
 'area_growth_rel_0_5h',
 'area_growth_rate_ha_per_h',
 'log1p_area_first',
 'log1p_growth',
 'log_area_ratio_0_5h',
 'relative_growth_0_5h',
 'radial_growth_m',
 'radial_growth_rate_m_per_h',
 'centroid_displacement_m',
 'centroid_speed_m_per_h',
 'spread_bearing_deg',
 'spread_bearing_sin',
 'spread_bearing_cos',
 'dist_std_ci_0_5h',
 'dist_change_ci_0_5h',
 'dist_slope_ci_0_5h',
 'closing_speed_m_per_h',
 'closing_speed_abs_m_per_h',
 'projected_advance_m',
 'dist_accel_m_per_h2',
 'dist_fit_r2_0_5h',
 'alignment_cos',
 'alignment_abs',
 'cross_track_component',
 'along_track_speed',
 'event_start_hour',
 'event_start_dayofweek',
 'event_start_month']

In [6]:
result = filter_features(
    enriched_train_df,
    base_features,
    enriched_features,
    model_type="rsf",
    n_repeats=10,
)
survival_features = result["survival_features"]  # TIER1 + TIER2
survival_features


Features — Base: 33 | Enriched: 47 | Total: 80
  Fold 1 | RSF baseline: 0.9642
  Fold 2 | RSF baseline: 0.9367
  Fold 3 | RSF baseline: 0.9643

  Permutation Importance [RSF]
#    Feature                              Importance   FoldStd  Type       Mark
1    log1p_dist_min                          0.01879   0.00281  enriched   ★
2    eta_along_track                         0.01580   0.00154  enriched   ★
3    eta_projected                           0.01279   0.00125  enriched   ★
4    eta_closing                             0.01277   0.00157  enriched   ★
5    eta_aligned_along                       0.01232   0.00212  enriched   ★
6    eta_radial                              0.01140   0.00453  enriched   ★
7    eta_combined                            0.00955   0.00336  enriched   ★
8    eta_bearing_adjusted                    0.00896   0.00496  enriched   ★
9    alignment_x_nperim                      0.00130   0.00146  enriched   +
10   dt_first_last_0_5h                      0.0011

['log1p_dist_min',
 'eta_along_track',
 'eta_projected',
 'eta_closing',
 'eta_aligned_along',
 'eta_radial',
 'eta_combined',
 'eta_bearing_adjusted',
 'alignment_x_nperim',
 'dt_first_last_0_5h',
 'alignment_abs',
 'dt_x_alignment',
 'dist_per_area',
 'low_temporal_resolution_0_5h']

In [9]:
# Multi-model
multi = filter_features_independent(
    enriched_train_df,
    base_features,
    enriched_features,
    n_repeats=10,
)
# → {"rsf": [...], "gradboost": [...], "coxnet": [...], "cgb": [...]}
multi


  Step 1/4: RSF Permutation Importance  [PRIMARY]

Features — Base: 33 | Enriched: 47 | Total: 80
  Fold 1 | RSF baseline: 0.9642
  Fold 2 | RSF baseline: 0.9367
  Fold 3 | RSF baseline: 0.9643

  Permutation Importance [RSF]
#    Feature                              Importance   FoldStd  Type       Mark
1    log1p_dist_min                          0.01879   0.00281  enriched   ★
2    eta_along_track                         0.01580   0.00154  enriched   ★
3    eta_projected                           0.01279   0.00125  enriched   ★
4    eta_closing                             0.01277   0.00157  enriched   ★
5    eta_aligned_along                       0.01232   0.00212  enriched   ★
6    eta_radial                              0.01140   0.00453  enriched   ★
7    eta_combined                            0.00955   0.00336  enriched   ★
8    eta_bearing_adjusted                    0.00896   0.00496  enriched   ★
9    alignment_x_nperim                      0.00130   0.00146  enriched   +


{'rsf': ['log1p_dist_min',
  'eta_along_track',
  'eta_projected',
  'eta_closing',
  'eta_aligned_along',
  'eta_radial',
  'eta_combined',
  'eta_bearing_adjusted',
  'alignment_x_nperim',
  'dt_first_last_0_5h',
  'alignment_abs',
  'dt_x_alignment',
  'dist_per_area',
  'low_temporal_resolution_0_5h'],
 'gradboost': ['log1p_dist_min',
  'eta_closing',
  'dist_per_area',
  'eta_aligned_along',
  'dt_first_last_0_5h',
  'eta_projected',
  'eta_along_track'],
 'coxnet': ['log1p_dist_min',
  'num_perimeters_0_5h',
  'dist_per_area',
  'log1p_area_first',
  'dt_first_last_0_5h',
  'alignment_abs',
  'area_speed_ratio',
  'eta_closing',
  'alignment_x_nperim',
  'dist_fit_r2_0_5h',
  'dist_cv',
  'cross_track_component',
  'eta_projected',
  'reliable_accel',
  'dt_x_alignment',
  'dist_change_norm',
  'directional_growth',
  'log1p_reliable_slope',
  'dist_slope_ci_0_5h',
  'growth_pressure',
  'spread_bearing_sin',
  'area_first_ha',
  'low_temporal_resolution_0_5h',
  'event_start_mon

### 6. Feature selection — inject custom wrapper

In [8]:
from xgbse import XGBSEDebiasedBCE
import numpy as np
from typing import Any
from dataclasses import field

class _XGBSurvivalFn:
    """
    Wrap one row of XGBSE predict DataFrame into a callable with .domain,
    similar to the sksurv step function interface that _survival_to_hit_probs expects.
    """
    __slots__ = ("times", "surv", "domain")

    def __init__(self, times: np.ndarray, surv: np.ndarray):
        self.times  = times
        self.surv   = np.clip(surv, 0.0, 1.0)
        self.domain = (float(times[0]), float(times[-1]))

    def __call__(self, t: float) -> float:
        return float(np.interp(t, self.times, self.surv,
                               left=self.surv[0], right=self.surv[-1]))

class XGBSEStrategy(ModelStrategy):
    """
    XGBSEDebiasedBCE survival strategy.
    Self resolve time_bins from training labels in fit().
    """

    def __init__(
        self,
        n_estimators:     int   = 200,
        learning_rate:    float = 0.05,
        max_depth:        int   = 4,
        subsample:        float = 0.8,
        colsample_bytree: float = 0.8,
        n_time_bins:      int   = 20,
    ):
        self.n_estimators     = n_estimators
        self.learning_rate    = learning_rate
        self.max_depth        = max_depth
        self.subsample        = subsample
        self.colsample_bytree = colsample_bytree
        self.n_time_bins      = n_time_bins
        self._time_bins       = None

    """
    XGBSEDebiasedBCE survival strategy.

    Self resolve time_bins from training labels in fit(),
    do not require user to specify
    """

    @property
    def name(self) -> str:
        return "XGBSE"

    # ── auto_params: scale by dataset size──────────────
    def auto_params(self, n: int) -> None:
        self.n_estimators = int(min(500, max(100, n * 1.0)))
        self.n_time_bins  = max(10, min(30, n // 10))

    # ── build: init model not fit ───────────────────────────────
    def build(self, random_state: int) -> XGBSEDebiasedBCE:
        return XGBSEDebiasedBCE(
            xgb_params={
                "n_estimators":     self.n_estimators,
                "learning_rate":    self.learning_rate,
                "max_depth":        self.max_depth,
                "subsample":        self.subsample,
                "colsample_bytree": self.colsample_bytree,
                "use_label_encoder": False,
                "eval_metric":      "logloss",
                "random_state":     random_state,
            }
        )

    # ── fit: resolve time_bins from label, then fit ─────────────────────
    def fit(self, model: XGBSEDebiasedBCE, X: np.ndarray, y: Any) -> Any:
        # y is structured array from Surv.from_dataframe
        # default field names of sksurv: y.dtype.names[0]=event, [1]=time
        event_field, time_field = y.dtype.names
        times  = y[time_field]
        events = y[event_field].astype(bool)

        # Use percentiles of event times to create bins (skip censored)
        event_times = times[events] if events.any() else times
        self._time_bins = np.unique(
            np.percentile(event_times,
                          np.linspace(0, 100, self.n_time_bins + 1))
        )

        model.fit(X, y, time_bins=self._time_bins)
        return model

    # ── predict_survival: return list callable with .domain ───────────────
    def predict_survival(self, model: XGBSEDebiasedBCE, X: np.ndarray) -> list:
        df: pd.DataFrame = model.predict(X)          # (n_samples, n_times)
        times = np.array(df.columns, dtype=float)    # time points from training

        # XGBSE returns survival P(T > t), NOT CDF
        # → use directly, do not need 1 - df
        surv_matrix = df.values                      # shape (n, T)

        return [
            _XGBSurvivalFn(times, surv_matrix[i])
            for i in range(len(surv_matrix))
        ]

    def summary_str(self) -> str:
        bins_str = (
            f"[{self._time_bins[0]:.0f}…{self._time_bins[-1]:.0f}]"
            if self._time_bins is not None else "not fit yet"
        )
        return (
            f"[XGBSE] n_est={self.n_estimators}  lr={self.learning_rate}  "
            f"depth={self.max_depth}  time_bins={len(self._time_bins) if self._time_bins is not None else '?'} {bins_str}"
        )

result = filter_features(
    enriched_train_df,
    base_features,
    enriched_features,
    n_repeats=10,
    wrapper=_TrainerWrapper(
        feature_cols=list(set(base_features + enriched_features)),
        strategy=XGBSEStrategy(n_estimators=200, learning_rate=0.05),
    ),
)


Features — Base: 33 | Enriched: 47 | Total: 80
  Fold 1 | XGBSE baseline: 0.9303
  Fold 2 | XGBSE baseline: 0.9212
  Fold 3 | XGBSE baseline: 0.9393

  Permutation Importance [XGBSE]
#    Feature                              Importance   FoldStd  Type       Mark
1    log1p_dist_min                          0.10721   0.09961  enriched   ★
2    eta_projected                           0.09873   0.12038  enriched   ★
3    eta_closing                             0.01204   0.01440  enriched   ★
4    dt_first_last_0_5h                      0.00606   0.00368  base       ★
5    eta_along_track                         0.00335   0.00580  enriched   +
6    dist_per_area                           0.00325   0.00299  enriched   +
7    num_perimeters_0_5h                     0.00233   0.00257  base       +
8    alignment_x_nperim                      0.00173   0.00283  enriched   +
9    area_speed_ratio                        0.00144   0.00128  enriched   +
10   alignment_abs                         